# 07 — Iterators vs Generators

This is the seventh and final notebook in `02_Iterators_and_Generators`.

The previous notebooks introduced:

```text
Iterable
   ↓
Iterator
   ↓
Iterator Protocol
   ↓
Custom Iterator
   ↓
Generator
   ↓
yield
   ↓
Generator Expression
```

Now the goal is to bring these ideas together and understand how **iterators and generators relate to each other**, how they differ in implementation, and when each approach is appropriate.

This notebook does not introduce another major mechanism. Instead, it consolidates the concepts from notebooks 01–06.

## 1. Introduction

By now, you have seen several ways to work with sequences of values:

- iterables provide data that can be iterated over
- iterators produce values one at a time
- the iterator protocol defines the behavior expected from an iterator
- custom iterators implement that protocol explicitly
- generators provide a simpler way to create iterators using `yield`
- generator expressions provide compact syntax for simple generators

The central relationship is:

```text
Generator
    ↓
is an
    ↓
Iterator
```

But an iterator does not have to be a generator.

## 2. Quick Review: Iterables

An **iterable** is an object from which an iterator can be obtained.

For example:

```python
numbers = [10, 20, 30]
```

A list is an iterable.

Use `iter()` to obtain an iterator:

```python
iterator = iter(numbers)
```

Then values can be requested from that iterator with `next()`.

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))

Key point:

> An iterable can provide an iterator.

The iterable itself does not necessarily represent the current iteration state. The iterator does.

## 3. Quick Review: Iterators

An **iterator** produces values one at a time and remembers its current position.

In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))

# The next call would raise StopIteration:
# print(next(iterator))

After all values have been produced, another `next()` call raises:

```text
StopIteration
```

An iterator follows the iterator protocol:

```python
__iter__()
__next__()
```

The protocol allows Python's `for` loop and `next()` to work consistently with iterator objects.

## 4. Quick Review: Generators

A generator function uses `yield`:

```python
def numbers():
    yield 10
    yield 20
    yield 30
```

Calling the function creates a generator object.

In [ ]:
def numbers():
    yield 10
    yield 20
    yield 30


generator = numbers()

print(next(generator))
print(next(generator))
print(next(generator))

Important:

> A generator is an iterator created through generator syntax.

Generators support `next()`, maintain their execution state, and raise `StopIteration` when they are exhausted.

## 5. Iterator vs Generator

This is the central comparison.

| Feature | Iterator | Generator |
|---|---|---|
| Produces values one at a time | Yes | Yes |
| Supports `next()` | Yes | Yes |
| Uses iterator protocol | Yes | Yes |
| Can be lazy | Yes | Yes |
| Usually created with | Class / `iter()` | `yield` / generator expression |
| Requires `__iter__()` / `__next__()` implementation | Custom iterators: yes | No |
| State management | Programmer manages it | Python manages generator state |
| Code complexity | Can be higher | Usually lower |

The crucial relationship is:

```text
Generator
    ↓
is an
    ↓
Iterator
```

But:

```text
Iterator
    ↓
is NOT necessarily
    ↓
Generator
```

## 6. How Iterators Are Created

There are two useful routes to remember.

### Built-in route

Many Python objects are iterable. Calling `iter()` on one creates an iterator:

In [ ]:
numbers = [1, 2, 3]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))

### Custom iterator

A custom iterator is a class that explicitly implements the iterator protocol.

In [ ]:
class CountUp:

    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value


counter = CountUp(3)

for number in counter:
    print(number)

This approach gives the programmer explicit control over the iteration state, but it requires more implementation code.

## 7. How Generators Are Created

The same count-up behavior can be written as a generator function:

In [ ]:
def count_up(limit):
    current = 1

    while current <= limit:
        yield current
        current += 1


for number in count_up(3):
    print(number)

The generator function does not need to define `__iter__()` or `__next__()` manually.

Python creates the generator object and manages the iterator behavior for you.

## 8. Iterator State vs Generator State

State is an important difference in how the two approaches are implemented.

### Custom iterator

In a custom iterator, the programmer explicitly stores state:

```python
self.current
```

For example:

In [ ]:
class CountUp:

    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value

### Generator

A generator can keep the state in an ordinary local variable:

```python
current = 1
```

Python preserves the generator's local execution state between `yield` statements.

In [ ]:
def count_up(limit):
    current = 1

    while current <= limit:
        yield current
        current += 1


generator = count_up(3)

print(next(generator))
print(next(generator))
print(next(generator))

The practical distinction is:

```text
Custom Iterator
    ↓
Programmer explicitly manages state

Generator
    ↓
Python preserves generator execution state
```

## 9. Lazy Evaluation

Both iterators and generators can produce values lazily.

Consider:

In [ ]:
def numbers():
    for number in range(1, 6):
        yield number


g = numbers()

print(next(g))
print(next(g))

Only the requested values have been consumed so far.

The important point is:

> Laziness is not exclusive to generators.

A custom iterator can also produce values one at a time and therefore behave lazily.

Generators are especially convenient because Python handles much of the state-management machinery automatically.

## 10. Custom Iterator vs Generator Function

Let's implement the same task in both forms.

### Custom iterator

In [ ]:
class Squares:

    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current ** 2
        self.current += 1
        return value


squares = Squares(5)

for value in squares:
    print(value)

### Generator function

In [ ]:
def squares(limit):
    for number in range(1, limit + 1):
        yield number ** 2


for value in squares(5):
    print(value)

The generator version is shorter because Python handles the iterator protocol and execution state.

The general pattern is:

```text
Custom Iterator
    ↓
More implementation control
    ↓
More code

Generator
    ↓
Automatic iterator machinery
    ↓
Less code
```

## 11. Code Comparison

Here is another direct comparison using even numbers.

### Iterator version

In [ ]:
class EvenNumbers:

    def __init__(self, limit):
        self.current = 0
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 2
        return value


iterator = EvenNumbers(10)

for number in iterator:
    print(number)

### Generator version

In [ ]:
def even_numbers(limit):
    current = 0

    while current <= limit:
        yield current
        current += 2


generator = even_numbers(10)

for number in generator:
    print(number)

Think about the comparison:

- Which version is shorter?
- Which version makes state management more explicit?
- Which version would you choose for a simple sequence?

For simple iteration logic, the generator is often easier to read and maintain.

## 12. Memory and Data Processing

Consider a list comprehension:

```python
numbers = [number ** 2 for number in range(1000000)]
```

This creates and stores all of the generated results in a list.

A generator expression behaves differently:

```python
numbers = (number ** 2 for number in range(1000000))
```

It produces values as they are requested.

Generators are therefore especially useful when:

- processing large sequences
- processing streams of data
- values do not need to be stored simultaneously
- only one pass through the data is required

The important distinction is:

> Generators can reduce memory usage because they produce values lazily, but not every generator is automatically faster.

This notebook focuses on the general behavior rather than benchmarking.

## 13. When to Use Iterators

Custom iterators can be useful when you need:

- explicit control over iteration state
- a reusable iterator class
- multiple configuration parameters
- more complex iterator behavior
- an object whose primary purpose is iteration

For example:

In [ ]:
class Countdown:

    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration

        value = self.current
        self.current -= 1
        return value


countdown = Countdown(5)

for number in countdown:
    print(number)

A custom iterator makes the state and iteration behavior explicit inside an object.

## 14. When to Use Generators

Generators are usually preferable when the iteration logic can be expressed naturally with `yield`.

Common examples include:

- sequences
- filtering
- transformations
- streaming values
- large data processing
- infinite sequences
- processing pipelines

For example:

In [ ]:
def positive_numbers(numbers):
    for number in numbers:
        if number > 0:
            yield number


values = [-5, 3, -2, 8, -1, 10]

for number in positive_numbers(values):
    print(number)

For a straightforward sequence or transformation, a generator often communicates the intent with less code than a custom iterator class.

## 15. Practical Examples

### Example 1 — Large sequence

In [ ]:
def squares(limit):
    for number in range(limit):
        yield number ** 2


for square in squares(10):
    print(square)

### Example 2 — Filtering

In [ ]:
def positive(values):
    for value in values:
        if value > 0:
            yield value


values = [-10, 5, -3, 8, 12, -1]

for value in positive(values):
    print(value)

### Example 3 — Infinite sequence

In [ ]:
def count():
    number = 1

    while True:
        yield number
        number += 1


numbers = count()

for _ in range(5):
    print(next(numbers))

An infinite generator must be consumed in a controlled way. The example above requests only five values.

### Example 4 — Generator expression

In [ ]:
squares = (number ** 2 for number in range(10))

for square in squares:
    print(square)

Generator expressions are another way to create generators when the required logic is simple enough to fit naturally into an expression.

## 16. Common Mistakes

### Mistake 1 — Thinking every iterator is a generator

False.

A list iterator:

In [ ]:
iterator = iter([1, 2, 3])

print(type(iterator))
print(next(iterator))

This object is an iterator, but it was not created by a generator function or generator expression.

### Mistake 2 — Thinking generators and iterators are completely different concepts

They are related.

A generator **is an iterator**.

That means a generator supports the iterator behavior you have already learned:

```python
next(generator)
```

and it can be used by a `for` loop.

### Mistake 3 — Assuming generators can be restarted

Generators are consumed.

In [ ]:
g = (x for x in range(3))

print(list(g))
print(list(g))

The second result is:

```python
[]
```

If you need to iterate again, create a new generator.

### Mistake 4 — Using a generator when repeated access is required

If you need direct access such as:

```python
values[5]
```

or need to iterate over stored values repeatedly, a list may be more appropriate.

Choose the data structure based on the operations your program needs.

### Mistake 5 — Choosing a custom iterator when a generator is much simpler

If all you need is:

```python
def numbers(limit):
    for number in range(limit):
        yield number
```

there is usually no reason to build a full iterator class just to produce this simple sequence.

Use a custom iterator when its explicit state or object-oriented behavior provides a real benefit.

## 17. Final Comparison

This table brings the major concepts together.

| Concept | Iterable | Iterator | Generator |
|---|---|---|---|
| Can be used with `for` | Yes | Yes | Yes |
| `iter()` works | Yes | Yes | Yes |
| `next()` works directly | Usually no | Yes | Yes |
| Remembers iteration state | Not necessarily | Yes | Yes |
| Implements iterator protocol | Not necessarily | Yes | Yes |
| Uses `yield` | No | No | Usually yes |
| Can be a custom class | Yes | Yes | Possible, but generator functions/expressions are typical |
| Lazy | Not necessarily | Usually | Yes |
| Can be infinite | Possible | Yes | Yes |

### The key relationship

```text
Iterable
   │
   │ iter()
   ▼
Iterator
   ▲
   │
Generator
```

More precisely:

```text
An iterable is something from which an iterator can be obtained.

An iterator produces values one at a time and follows
the iterator protocol.

A generator is an iterator created through generator syntax.

A generator is an iterator.
An iterator is not necessarily a generator.
```

## 18. Summary

### Key Takeaways

- Iterable, iterator, and generator are related but different concepts.
- An iterable can provide an iterator.
- An iterator produces values one at a time.
- A generator is a type of iterator.
- Generators use `yield` to produce values lazily.
- Custom iterators require explicit iterator-protocol implementation.
- Generators usually require much less code.
- Both iterators and generators can maintain state.
- Both can be lazy.
- Generators are useful for large or streaming data.
- Custom iterators are useful when explicit iteration behavior is needed.
- Generator expressions provide a concise way to create generators.
- Generators are not automatically faster; their major advantage here is often lazy production and reduced storage of intermediate values.

### Complete `02_Iterators_and_Generators` Progression

```text
02_Iterators_and_Generators
│
├── 01_Iterables_and_Iterators.ipynb
├── 02_Iterator_Protocol.ipynb
├── 03_Custom_Iterators.ipynb
├── 04_Generators.ipynb
├── 05_Yield_and_Generator_Functions.ipynb
├── 06_Generator_Expressions.ipynb
└── 07_Iterators_vs_Generators.ipynb
```

This completes the **Iterators & Generators** section.

The next major section in the Advanced roadmap is:

```text
03_Decorators
```